<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase36_COLAB_DOWNLOAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 36 — Source-verified method freeze and future four-arm experiment contract
**Google Drive → Google Drive · fully synthetic · runs without Athena**

**Research question.** Phase 35 reconstructed the 19-field *extracted source manifest*, but a reconstruction can match a wrong manifest perfectly. This phase independently verifies all 148 manifest rows against the **original, exact-SHA Phase 33 FHIR JSON**, checks 10 prespecified manifest corruptions, and freezes a field-by-field standards and experimental readiness contract. It distinguishes evidence available in the actual Phase 33 synthetic FHIR source from the earlier Phase 29 manufactured procedure test fixtures, which do **not** apply to these 19 new people.

**Inputs (all from previous Colab Drive runs):** Phase 35 shareable + seven SHA-tagged output artifacts; Phase 34 shareable + canonical source manifest; original Phase 33 synthetic FHIR bundle. Optional: the exact earlier Phase 27 imaging schema, Phase 28 OMOP 5.5 core schema, Phase 31 public binding status, and a *private* Athena checklist. Hash-matched variants such as `(...)(1).csv` are accepted. A mix of unrelated runs is rejected. Set `DRIVE_RELATIVE_FOLDER` if you have multiple separate project folders.

**Crucial limitations:** No real ADNI/MRI, real DICOM UID attestation, external clinical ground truth, pyOMOP ETL, official MI-CDM loading, approved clinical vocabulary or four-arm outcome comparison. All people, UIDs and observations are synthetic. A successful raw-source audit is a data-integrity check, **not** Arm A completion or proof of novel scientific contribution. A descriptive 9-person synthetic holdout is not an external performance estimate. Original generic TFL Arm C remains nonportable/unexecuted.

In [9]:
import os
DRIVE_ROOT_OVERRIDE=os.environ.get('ICHI2027_DRIVE_ROOT','').strip()
if not DRIVE_ROOT_OVERRIDE:
    from google.colab import drive
    drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import csv,json,hashlib,copy,re,os
from pathlib import Path
from decimal import Decimal
from collections import Counter,defaultdict
from datetime import datetime,timezone
DRIVE_ROOT=Path(DRIVE_ROOT_OVERRIDE) if DRIVE_ROOT_OVERRIDE else Path('/content/drive/MyDrive')
DRIVE_RELATIVE_FOLDER=''   # e.g. 'NeuroFHIR_ICHI2027'; leave blank for all MyDrive
ROOT=DRIVE_ROOT/DRIVE_RELATIVE_FOLDER if DRIVE_RELATIVE_FOLDER else DRIVE_ROOT
assert ROOT.is_dir(), 'Drive not found: '+str(ROOT)
PH35='35_person_disjoint_synthetic_transport_generalization_and_mixed_study_hybrid'
PH34='34_adaptive_synthetic_evidence_carrier_prototype'
IMAGEWG='8f0a19bb94e941bb335a097bc9851fab672621c7'
OMOP55='4a910305b2cb74a4fc2b2c34baf44eb0542ff03f'

def sha(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for buf in iter(lambda:f.read(1<<20),b''):h.update(buf)
 return h.hexdigest()
def canon(v):return json.dumps(v,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def csvread(p):
 with Path(p).open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
def csvsave(p,rows,fields):
 with Path(p).open('w',encoding='utf-8',newline='') as f:
  wr=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore');wr.writeheader();wr.writerows(rows)
def name_match(real,basename):
 stem,ext=os.path.splitext(basename)
 return real==basename or bool(re.fullmatch(re.escape(stem)+r'\(\d+\)'+re.escape(ext),real))
def walk(root):
 for directory,ds,fs in os.walk(str(root),onerror=lambda exc:None):
  ds[:]=[d for d in ds if not d.startswith('.') and d not in {'_PRIVATE_ATHENA','node_modules','.git'}]
  for file in sorted(fs):yield Path(directory)/file
def find_hash(target_sha,basename=None,folder=None):
 paths=(folder.iterdir() if folder else walk(ROOT))
 for p in paths:
  try:
   if p.is_file() and (basename is None or name_match(p.name,basename)) and sha(p)==target_sha:return p
  except (OSError,FileNotFoundError):continue
 return None

def matched_run(phase,basename,required):
 issues=[]
 for jp in walk(ROOT):
  if not name_match(jp.name,basename):continue
  try:
   record=json.loads(jp.read_text(encoding='utf-8'))
   if record.get('phase')!=phase:continue
   local={key:find_hash(record['artifact_sha256'][key],name,jp.parent) for key,name in required.items()}
   if all(local.values()):return jp,record,local
   issues.append({'candidate':str(jp),'missing_or_mismatch':[k for k,v in local.items() if v is None]})
  except (ValueError,OSError,KeyError) as exc:issues.append(str(exc))
 raise FileNotFoundError('No self-consistent '+phase+' artifact set: '+str(issues[:4]))

REQ35={
 'split':'ICHI2027_Phase35_PERSON_DISJOINT_SPLIT.csv',
 'plans':'ICHI2027_Phase35_PLAN_SPLIT_EVALUATION.csv',
 'scenario_plans':'ICHI2027_Phase35_SCENARIO_PLAN_AUDIT.csv',
 'scenarios':'ICHI2027_Phase35_STRESS_SCENARIOS.csv',
 'tests':'ICHI2027_Phase35_RECONSTRUCTION_TESTS.csv',
 'policy':'ICHI2027_Phase35_SYNTHETIC_PROXY_PRESERVATION.csv',
 'method':'ICHI2027_Phase35_METHOD_SPECIFICATION.json'
}
J35,P35,F35=matched_run(PH35,'ICHI2027_Phase35_SHAREABLE.json',REQ35)
assert P35['upstream']['imagewg_commit']==IMAGEWG and P35['upstream']['omop55_commit']==OMOP55
assert P35['tests']['checks_passing']==P35['tests']['synthetic_checks']==14
assert P35['gates']['source_supported_19_bindings_approved'] is False
J34=find_hash(P35['upstream']['phase34_shareable_sha256'],'ICHI2027_Phase34_SHAREABLE.json')
assert J34, 'Exact upstream Phase34 shareable not found'
P34=json.loads(J34.read_text(encoding='utf-8'))
assert P34['phase']==PH34
SOURCE_PATH=find_hash(P35['upstream']['phase34_source_sha256'],'ICHI2027_Phase34_CANONICAL_SOURCE_MANIFEST.csv',J34.parent)
assert SOURCE_PATH, 'Missing exact Phase34 canonical source CSV'
FHIR_PATH=find_hash(P34['upstream']['phase33_bundle_sha256'],'ICHI2027_Phase33_NEW_SYNTHETIC_FHIR_BUNDLE.json')
assert FHIR_PATH, ('Missing exact original Phase33 synthetic FHIR JSON: copy its original Colab output to MyDrive; '
                  'never use a Phase32 or Phase29 bundle as substitute.')
SOURCE=csvread(SOURCE_PATH)
FHIR=json.loads(FHIR_PATH.read_text(encoding='utf-8'))
assert FHIR['resourceType']=='Bundle' and FHIR['type']=='collection'
assert len(SOURCE)==148 and len({r['observation_id'] for r in SOURCE})==148
assert P35['person_disjoint_synthetic_transfer']['holdout_people']==9
assert P34['method']['selected_plan']=='VISIT_CODED'
OUT_DIR=J35.parent
print('Verified Phase35:',J35.name)
print('Verified Phase34 source SHA:',sha(SOURCE_PATH))
print('Verified original Phase33 FHIR SHA:',sha(FHIR_PATH))
print('Output folder:',OUT_DIR)

Verified Phase35: ICHI2027_Phase35_SHAREABLE.json
Verified Phase34 source SHA: 4f10ec138f32108c52ce5d29c2c51a11ecaca27bce1b5d6946187ec484b2d2ba
Verified original Phase33 FHIR SHA: 4d2e64b3dad926f9a32fc2d243f8ebb04dcff0159d60c5343f01e0a1985d1bf4
Output folder: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs


## 1 · Independent raw-FHIR source reconciliation
The Phase 34 manifest is *not* treated as ground truth. Each source-field claim is compared with the referenced FHIR Observation, ImagingStudy, Device and Provenance from the unchanged Phase 33 JSON. Missing or duplicate references fail. The audit covers 18 independently observable fields per row (the 19th is a derived opaque observation key). It is intentionally **not** a statistical test of clinical measurement quality.

In [11]:
# Index actual original FHIR entities without using the extracted manifest as a lookup oracle.
E=defaultdict(dict)
for entry in FHIR['entry']:
 r=entry['resource']; kind=r['resourceType']; ident=r['id']
 assert ident not in E[kind],f'Duplicate original FHIR {kind}/{ident}'
 E[kind][ident]=r
assert {k:len(E[k]) for k in ['Patient','Device','ImagingStudy','Observation','Provenance']}=={
 'Patient':19,'Device':19,'ImagingStudy':38,'Observation':148,'Provenance':148}
assert len(E.get('Procedure',{}))==0, 'Unexpected source procedure: revise source evidence contract before continuing'
SOURCE_FIELDS=['source_observation_ref','person_ref','visit','effective_datetime','value_raw',
 'study_ref','study_uid','series_uid','study_started','modality','device_ref','region',
 'side','unit','trace_state','trace_rule','provenance_id','provenance_sources']
assert set(SOURCE[0])==set(SOURCE_FIELDS)|{'observation_id'}

def leaf(ref,expected):
 prefix=expected+'/'
 assert isinstance(ref,str) and ref.startswith(prefix) and len(ref)>len(prefix),f'Bad {expected} reference: {ref}'
 return ref[len(prefix):]
def ext(resource,suffix,key):
 matches=[x[key] for x in resource.get('extension',[]) if x.get('url','').endswith('/'+suffix) and key in x]
 assert len(matches)==1, f'FHIR extension {suffix} must appear exactly once on {resource.get("id")}'
 return str(matches[0])
def expected_from_fhir(source_observation_ref):
 oid=leaf(source_observation_ref,'Observation')
 obs=E['Observation'][oid]
 study_ref=obs['focus'][0]['reference']; study=E['ImagingStudy'][leaf(study_ref,'ImagingStudy')]
 device_ref=obs['device']['reference']; device=E['Device'][leaf(device_ref,'Device')]
 assert device['resourceType']=='Device'
 person_ref=obs['subject']['reference']; assert person_ref==study['subject']['reference']
 assert leaf(person_ref,'Patient') in E['Patient']
 pv=[r for r in E['Provenance'].values() if any(t.get('reference')==source_observation_ref for t in r.get('target',[]))]
 assert len(pv)==1, 'Expected one and only one source Provenance per FHIR Observation'
 pr=pv[0]
 sources=sorted(x['what']['reference'] for x in pr.get('entity',[]) if x.get('role')=='source')
 assert study_ref in sources and device_ref in sources
 code=obs['code']['coding'][0]['code']
 assert code in ['hippocampus-volume','entorhinal-volume']
 region=code.split('-volume')[0]
 an=obs['bodySite']['coding'][0]['code']
 assert an in ['SYN-HIP-LEFT','SYN-HIP-RIGHT','SYN-ENT-LEFT','SYN-ENT-RIGHT']
 assert (region=='hippocampus')==(an.startswith('SYN-HIP-'))
 side=an.rsplit('-',1)[-1].lower()
 uu=[v['value'] for v in study.get('identifier',[]) if v.get('system')=='urn:dicom:uid']
 assert len(uu)==1 and uu[0].startswith('urn:oid:2.25.')
 assert len(study['series'])==1, 'Multiple series require a separately reviewed source attribution rule'
 se=study['series'][0]
 assert len(study['series'])==1
 return {
  'source_observation_ref':source_observation_ref,'person_ref':person_ref,
  'visit':ext(obs,'timepoint-label','valueString'),
  'effective_datetime':obs['effectiveDateTime'],'value_raw':str(obs['valueQuantity']['value']),
  'study_ref':study_ref,'study_uid':uu[0].removeprefix('urn:oid:'),
  'series_uid':se['uid'],'study_started':study['started'],
  'modality':se['modality']['code'],'device_ref':device_ref,
  'region':region,'side':side,'unit':obs['valueQuantity']['unit'],
  'trace_state':ext(obs,'trace-state','valueCode'),
  'trace_rule':ext(obs,'trace-rule-version','valueString'),
  'provenance_id':'Provenance/'+pr['id'],'provenance_sources':sources
 }

def compare_one(row):
 ref=row.get('source_observation_ref','')
 try: exp=expected_from_fhir(ref)
 except (AssertionError,KeyError,IndexError,TypeError,ValueError) as er:return ['INVALID_FHIR_REFERENCE_OR_SOURCE: '+str(er)]
 errs=[]
 for field in SOURCE_FIELDS:
  val=row.get(field)
  if field=='provenance_sources':
   try:val=sorted(json.loads(val) if isinstance(val,str) else val)
   except (ValueError,TypeError):val='INVALID_JSON'
  if field=='value_raw':
   try:equal=Decimal(str(val))==Decimal(str(exp[field]))
   except Exception:equal=False
  else:equal=val==exp[field]
  if not equal:errs.append('MISMATCH_'+field)
 return errs

AUDIT=[]
for row in SOURCE:
 errors=compare_one(row)
 AUDIT.append({'source_observation_ref':row['source_observation_ref'],
  'field_checks':len(SOURCE_FIELDS),'mismatch_count':len(errors),'status':'PASS' if not errors else 'FAIL',
  'mismatch_fields':'|'.join(errors)})
assert {r['source_observation_ref'] for r in SOURCE}=={'Observation/'+x for x in E['Observation']}
assert len(AUDIT)==148 and all(x['status']=='PASS' for x in AUDIT), 'Phase34 manifest differs from original Phase33 FHIR; inspect audit before interpreting prior results.'
assert len({r['person_ref'] for r in SOURCE})==19
assert len({r['study_ref'] for r in SOURCE})==38
FIELD_CHECK_COUNT=sum(x['field_checks'] for x in AUDIT)
assert FIELD_CHECK_COUNT==148*18
print(f'Original FHIR independently verifies all 148 rows and {FIELD_CHECK_COUNT} selected source fields.')

Original FHIR independently verifies all 148 rows and 2664 selected source fields.


## 2 · Ten prespecified negative controls
Corrupt exactly one extracted manifest field at a time while keeping original Phase 33 FHIR unchanged. Confirm that comparison detects the *right changed field*. This specifically addresses the risk that the encoder and decoder agree with each other but both inherit an incorrect source extraction.

In [12]:
NEGATIVE=[]
first=next(x for x in SOURCE if x['side']=='left')
def mutation(name,field,new,expected):
 changed=copy.deepcopy(first)
 changed[field]=new(changed) if callable(new) else new
 errs=compare_one(changed)
 success=expected in errs and len(errs)==1
 NEGATIVE.append({'test':name,'mutated_field':field,'expected':expected,
                  'observed':'|'.join(errs),'passed':success})
mutation('quantity_changed','value_raw',lambda r:str(Decimal(r['value_raw'])+1),'MISMATCH_value_raw')
mutation('laterality_swapped','side','right','MISMATCH_side')
mutation('study_UID_swapped','study_uid','2.25.999','MISMATCH_study_uid')
mutation('provenance_study_removed','provenance_sources',lambda r:canon([r['device_ref']]),'MISMATCH_provenance_sources')
mutation('TRACE_rule_changed','trace_rule','TRACE-synthetic-v999','MISMATCH_trace_rule')
mutation('cross_person_attribution','person_ref','Patient/held33-p019','MISMATCH_person_ref')
mutation('wrong_time','effective_datetime','2099-01-01T00:00:00Z','MISMATCH_effective_datetime')
mutation('wrong_modality','modality','CT','MISMATCH_modality')
mutation('unresolved_study_link','study_ref','ImagingStudy/missing','MISMATCH_study_ref')
mutation('changed_unit','unit','mL','MISMATCH_unit')
assert len(NEGATIVE)==10 and all(x['passed'] for x in NEGATIVE),NEGATIVE
print('All 10 deliberate extraction faults caught against unchanged original FHIR.')

All 10 deliberate extraction faults caught against unchanged original FHIR.


## 3 · Reconcile Phase 35's benchmark without redefining its holdout
The plan was already selected on the 10-person training subset. Here the registered Phase 35 CSVs are checked against their own SHA-backed report, with no reoptimization and no attempt to convert synthetic results into clinical performance claims. Exclude source-conflicting UID quarantine scenarios from compression denominators.

In [13]:
SPLIT=csvread(F35['split']);PLANS=csvread(F35['plans']);STRESS=csvread(F35['scenarios'])
STRESS_PLANS=csvread(F35['scenario_plans']);TESTS=csvread(F35['tests']);POLICY=csvread(F35['policy'])
METHOD=json.loads(F35['method'].read_text(encoding='utf-8'))
assert len(SPLIT)==19 and Counter(r['partition'] for r in SPLIT)=={'TRAIN':10,'HOLDOUT':9}
assert len({r['synthetic_person_ref'] for r in SPLIT})==19
assert {r['synthetic_person_ref'] for r in SPLIT}=={r['person_ref'] for r in SOURCE}
assert sum(r['partition']=='TRAIN' for r in SPLIT)==10
assert len(TESTS)==14 and all(r['passed'].lower()=='true' for r in TESTS)
assert len(STRESS)==6 and {r['scenario'] for r in STRESS}=={
 'HOLDOUT_CLEAN','MIXED_STUDY_SAME_VISIT','MISSING_FOLLOWUP_HEMISPHERE',
 'UNSEEN_ANATOMY_CODE','FOUR_MIXED_VISITS','CONFLICTING_STUDY_UID_QUARANTINE'}
assert METHOD['version']=='P35_v1' and METHOD['plan_selection'].startswith('smallest canonical JSON')
holdout={r['synthetic_person_ref'] for r in SPLIT if r['partition']=='HOLDOUT'}
assert sum(r['person_ref'] in holdout for r in SOURCE)==69
assert P35['person_disjoint_synthetic_transfer']['selected_train_plan']=='VISIT_CODED'
assert P35['person_disjoint_synthetic_transfer']['holdout_reduction_vs_inline_percent']==38.4897
brows=[]
for r in STRESS:
 name=r['scenario']; quarantined=name=='CONFLICTING_STUDY_UID_QUARANTINE'
 if quarantined:
  assert r['fallback_status']=='QUARANTINED_SOURCE_NOT_ENCODED' and r['deployed_plan']=='NONE'
  brows.append({'scenario':name,'n_people':r['synthetic_people'],'n_observations':r['synthetic_observations'],
    'status':'SOURCE_CONFLICT_QUARANTINED','deployed_plan':'NONE','reduction_percent':'NOT_APPLICABLE',
    'oracle_comparison':'NOT_APPLICABLE'})
  continue
 assert r['source_vs_restored_proxy_person_agreement']=='9'
 assert float(r['deployed_saving_vs_inline_percent'])>0
 prow=[x for x in STRESS_PLANS if x['scenario']==name and x['plan']==r['deployed_plan']]
 assert len(prow)==1 and prow[0]['admissible'].lower()=='true'
 assert int(prow[0]['selected_field_roundtrip'])==int(r['synthetic_observations'])
 assert int(prow[0]['metadata_bytes'])==int(r['deployed_metadata_bytes'])
 brows.append({'scenario':name,'n_people':r['synthetic_people'],'n_observations':r['synthetic_observations'],
  'status':'SYNTHETIC_SELECTED_FIELDS_ROUNDTRIP','deployed_plan':r['deployed_plan'],
  'reduction_percent':r['deployed_saving_vs_inline_percent'],
  'oracle_comparison':'POSTHOC_DESCRIPTIVE_ONLY'})
assert len(POLICY)==9*5  # conflicting UID scenario is quarantined and has no cohort rows
assert all(x['source_restored_exact_agreement'].lower()=='true' for x in POLICY if x['scenario']!='CONFLICTING_STUDY_UID_QUARANTINE')
print('Frozen 10/9 split, 14 prior tests and six scenario summaries verified; no refitting.')

Frozen 10/9 split, 14 prior tests and six scenario summaries verified; no refitting.


## 4 · Source-availability and official target-field evidence matrix
An exact pinned field inventory is loaded **only if its original byte hash is available in Drive**. The source-side audit always runs. Missing pinned CSVs yield an explicit unavailable schema gate; the notebook never invents replacement official field rows. For the new 19-person Phase 33 cohort, no FHIR `Procedure`, sex, birth date, race or ethnicity has been supplied. The four manufactured Phase 29 `Procedure` records belong to different test people and cannot be transferred to this cohort.

In [14]:
# These byte digests were pinned in prior phase results, not computed using altered local copies.
IMAGING_FIELDS_SHA='11171354f1e916efa4dccbf974ba19a41b4ea6b4ef03bf6926a74133dd59c9f8'
CORE_FIELDS_SHA='ea14b96b2e1560ff11bae62003d18ebee619c0c4081044b249fdf22d227b8b3b'
imaging_csv=find_hash(IMAGING_FIELDS_SHA,'ICHI2027_Phase27_REPAIRED_PINNED_TARGET_FIELDS.csv')
core_csv=find_hash(CORE_FIELDS_SHA,'ICHI2027_Phase28_PINNED_OMOP55_CORE_FIELDS.csv')
schema_present=bool(imaging_csv and core_csv)
SCHEMA_ROWS=[]
if imaging_csv:
 for r in csvread(imaging_csv):SCHEMA_ROWS.append({'table':r['table'].lower(),'field':r['field'], 'required':r['required'],'source':'PINNED_IMAGEWG'})
if core_csv:
 for r in csvread(core_csv):SCHEMA_ROWS.append({'table':r['table'].lower(),'field':r['field'],'required':r['required'],'source':'PINNED_OMOP55'})
assert not imaging_csv or len([r for r in SCHEMA_ROWS if r['source']=='PINNED_IMAGEWG'])==23
assert not core_csv or len([r for r in SCHEMA_ROWS if r['source']=='PINNED_OMOP55'])==58
assert len({(r['table'],r['field']) for r in SCHEMA_ROWS})==len(SCHEMA_ROWS)
PAT=E['Patient']; OBS=E['Observation']; IMG=E['ImagingStudy']; PRO=E.get('Procedure',{})
assert len(PAT)==19 and len(IMG)==38 and len(OBS)==148 and not PRO
assert sum(bool(p.get('gender')) for p in PAT.values())==0
assert sum(bool(p.get('birthDate')) for p in PAT.values())==0
assert sum(bool(p.get('extension')) for p in PAT.values())==0
SOURCE_STATUS={
 ('person','person_id'):'SYNTHETIC_STABLE_KEY_GENERATABLE',
 ('person','year_of_birth'):'BLOCKED_NO_FHIR_BIRTHDATE',
 ('person','gender_concept_id'):'BLOCKED_NO_SOURCE_GENDER_AND_NO_REVIEWED_VOCAB',
 ('person','race_concept_id'):'BLOCKED_NO_SOURCE_RACE_AND_NO_POLICY',
 ('person','ethnicity_concept_id'):'BLOCKED_NO_SOURCE_ETHNICITY_AND_NO_POLICY',
 ('procedure_occurrence','procedure_occurrence_id'):'BLOCKED_NO_PHASE33_SOURCE_PROCEDURE',
 ('procedure_occurrence','person_id'):'BLOCKED_NO_PHASE33_SOURCE_PROCEDURE',
 ('procedure_occurrence','procedure_concept_id'):'BLOCKED_NO_PHASE33_SOURCE_PROCEDURE_AND_VOCAB',
 ('procedure_occurrence','procedure_type_concept_id'):'BLOCKED_NO_PHASE33_SOURCE_PROCEDURE_AND_VOCAB',
 ('procedure_occurrence','procedure_date'):'BLOCKED_NO_PHASE33_SOURCE_PROCEDURE',
 ('image_occurrence','image_occurrence_id'):'SYNTHETIC_STABLE_KEY_GENERATABLE',
 ('image_occurrence','person_id'):'SOURCE_PRESENT_SYNTHETIC',
 ('image_occurrence','procedure_occurrence_id'):'BLOCKED_NO_PHASE33_SOURCE_PROCEDURE',
 ('image_occurrence','image_occurrence_date'):'SOURCE_PRESENT_SYNTHETIC',
 ('image_occurrence','image_study_uid'):'SYNTHETIC_UID_NOT_DICOM_ATTESTED',
 ('image_occurrence','image_series_uid'):'SYNTHETIC_UID_NOT_DICOM_ATTESTED',
 ('image_occurrence','modality_concept_id'):'BLOCKED_ATHENA_VOCAB_SOURCE_MR_PRESENT',
 ('image_feature','image_feature_id'):'SYNTHETIC_STABLE_KEY_GENERATABLE',
 ('image_feature','person_id'):'SOURCE_PRESENT_SYNTHETIC',
 ('image_feature','image_occurrence_id'):'SOURCE_PRESENT_SYNTHETIC',
 ('image_feature','image_feature_concept_id'):'BLOCKED_ATHENA_ANATOMY_LATERALITY_CONCEPT',
 ('image_feature','image_feature_type_concept_id'):'BLOCKED_ATHENA_FEATURE_TYPE_CONCEPT',
 ('image_feature','image_feature_event_field_concept_id'):'BLOCKED_EVENT_FIELD_METADATA_ADJUDICATION',
 ('image_feature','image_feature_event_id'):'SIDECAR_ONLY_EVENT_FIELD_UNVALIDATED',
 ('image_feature','alg_system'):'SOURCE_SYNTHETIC_DEVICE_ID_AND_VERSION',
 ('measurement','measurement_id'):'SYNTHETIC_STABLE_KEY_GENERATABLE',
 ('measurement','person_id'):'SOURCE_PRESENT_SYNTHETIC',
 ('measurement','measurement_concept_id'):'BLOCKED_ATHENA_LATERALIZED_MEASUREMENT',
 ('measurement','measurement_type_concept_id'):'BLOCKED_ATHENA_MEASUREMENT_TYPE',
 ('measurement','measurement_date'):'SOURCE_PRESENT_SYNTHETIC',
 ('measurement','value_as_number'):'SOURCE_PRESENT_SYNTHETIC',
 ('measurement','unit_concept_id'):'BLOCKED_ATHENA_UNIT_MM3',
}
FIELD_MATRIX=[]
for x in SCHEMA_ROWS:
 status=SOURCE_STATUS.get((x['table'],x['field']),'NOT_ASSESSED_IN_THIS_PHASE')
 FIELD_MATRIX.append({**x,'evidence_status':status,
  'synthetic_source_cohort':'PHASE33_19_PERSONS_NO_PROCEDURES',
  'armA_field_populated':False})
assert not schema_present or len(FIELD_MATRIX)==81
print('Official pinned schemas found:',schema_present,'exact fields:',len(FIELD_MATRIX))
print('Phase33 source procedures:',len(PRO),'; gender present:',0,'; birthdate present:',0)

Official pinned schemas found: True exact fields: 81
Phase33 source procedures: 0 ; gender present: 0 ; birthdate present: 0


## 5 · Public binding gate and registration of future experimental arms
If the Phase 31 public status CSV is available, use its 19 distinct binding keys, **not** suggested concept IDs. A separately authorized, frozen `CONCEPT.csv` and independently completed private review are still necessary. No private Athena records or researcher identifiers are exported. Original gap count 36 belongs to the earlier 2-person Phase 29 synthetic fixture; the latest 19-person source has additional unlinked procedures and missing demographics and cannot inherit those fixture resolutions.

In [15]:
P31_STATUS=None
for p in walk(ROOT):
 if name_match(p.name,'ICHI2027_Phase31_BINDING_REVIEW_STATUS.csv'):
  try:
   candidate=csvread(p)
   if len(candidate)==19 and len({x['binding_key'] for x in candidate})==19:
    P31_STATUS=candidate;break
  except (OSError,ValueError,KeyError):pass
BASE_KEYS=['IO_MODALITY_MR','IF_HIP_LEFT','IF_HIP_RIGHT','IF_ENT_LEFT','IF_ENT_RIGHT',
 'IF_TYPE_ALGORITHM','PERSON_GENDER_F','PERSON_GENDER_M','PERSON_RACE_UNRECORDED',
 'PERSON_ETHNICITY_UNRECORDED','PROCEDURE_MRI','PROCEDURE_TYPE_SYN','MEAS_HIP_LEFT',
 'MEAS_HIP_RIGHT','MEAS_ENT_LEFT','MEAS_ENT_RIGHT','MEAS_TYPE_SYN','UNIT_MM3','IF_EVENT_MEAS_ID']
if P31_STATUS:assert set(BASE_KEYS)=={x['binding_key'] for x in P31_STATUS}
PRIVATE=OUT_DIR/'_PRIVATE_ATHENA';PRIVATE.mkdir(exist_ok=True)
PRIVATE_GUIDE=PRIVATE/'PHASE36_ATHENA_AND_REVIEW_CHECKLIST.txt'
if not PRIVATE_GUIDE.exists():
 PRIVATE_GUIDE.write_text("\n".join([
  'PHASE36 PRIVATE PREREQUISITES (DO NOT COMMIT / DO NOT UPLOAD TO PUBLIC GITHUB)',
  '',
  '1. Obtain an authorized OHDSI Athena CONCEPT.csv export; keep it under this _PRIVATE_ATHENA folder.',
  '2. Keep the exact export date, release and SHA-256 in a private manifest.',
  '3. Open the Phase31 private 19-row reviewed-binding worksheet and independently choose each concept.',
  '   Document semantic justification, domain, evidence and review date.',
  '4. Missing gender and birthdate in Phase33 synthetic FHIR cannot be recovered through a vocabulary.',
  '5. Phase33 has zero FHIR Procedure records. For synthetic mapping only, create explicitly marked fixtures.',
  '   Do not borrow unrelated Phase29 procedures or present fixtures as real clinical evidence.',
  '6. Independently adjudicate IMAGE_FEATURE event-field metadata; invent no concept IDs.',
  '7. Re-run Phase31 private validation with frozen CONCEPT.csv and reviewed bindings.',
  '   This notebook never approves clinical concepts just because an export exists.',
 ]),encoding="utf-8")
BINDING=[]
for key in BASE_KEYS:
 prior=next((r for r in (P31_STATUS or []) if r['binding_key']==key),None)
 BINDING.append({'binding_key':key,'phase31_public_status':prior['verification_status'] if prior else 'NOT_LOADED',
  'phase35_binding_gate':'NOT_APPROVED','required_action':
   'REVIEW_SOURCE_AND_ATHENA' if key not in ['PERSON_GENDER_F','PERSON_GENDER_M','PERSON_RACE_UNRECORDED','PERSON_ETHNICITY_UNRECORDED','IF_EVENT_MEAS_ID'] else
   ('SOURCE_GENDER_ABSENT_NO_SEX_MAPPING' if key.startswith('PERSON_GENDER') else
    'REVIEW_MISSINGNESS_POLICY_AND_VOCAB' if key.startswith('PERSON_') else
    'ADJUDICATE_METADATA_EVENT_FIELD')})
assert len(BINDING)==19
ARM=[
 {'arm':'A','description':'Faithful standards-supported FHIR to OMOP v5.5 + pinned ImageWG MI-CDM',
  'status':'BLOCKED','reason':'No reviewed vocabulary, Phase33 source Procedure or birthdate; no official ETL'},
 {'arm':'B','description':'Deliberate value-only ablation',
  'status':'SYNTHETIC_SHADOW_ONLY','reason':'Phase34 value-only comparator is not an official CDM-load result'},
 {'arm':'C','description':'Original frozen generic TFL v15',
  'status':'NONPORTABLE_NOT_EXECUTED','reason':'Freeze original generic method; no claims of imaging performance'},
 {'arm':'D','description':'Adaptive minimal evidence carrier and hybrid fallback',
  'status':'SYNTHETIC_ALGORITHMIC_PROTOTYPE','reason':'Phase34/35 exact selected-field tests; novelty and external generalization unverified'},
]
PROTOCOL={
 'protocol_id':'ICHI2027_P36_FOUR_ARM_FREEZE_V1',
 'status':'INTERNAL_VERSIONED_FUTURE_PROTOCOL_NOT_PUBLICLY_PREREGISTERED',
 'registration_boundary':{
   'historical_results':'Phase34 and Phase35 synthetic transport results occurred before this protocol; they are retrospective feasibility outcomes and MUST NOT be called prospectively preregistered.',
   'future_claim':'Prospective protocol applies only to new future study data and validation decisions collected after this file and its SHA-256 are frozen.',
   'algorithm_spec_sha256':P35['artifact_sha256']['method'],
   'frozen_phase35_shareable_sha256':sha(J35),
   'original_fhir_bundle_sha256':sha(FHIR_PATH),
   'full_executable_algorithm_independently_archived':False,
   'source_or_method_change_rule':'Any subsequent changes create a newly numbered protocol, preserve this JSON hash and explain deviations.'},
 'research_question':'Do source-faithful standards mapping and an adaptive evidence carrier preserve clinically relevant, laterality-specific longitudinal MRI evidence and cohort membership compared with deliberately value-only transport and the original generic TFL v15?',
 'study_design':{
   'historical_synthetic_benchmark':'19 synthetic people, 148 observations, original SHA-backed Phase35 10-person training / 9-person holdout; descriptive prior work, not independent external data.',
   'future_external_independence':'New cases independently authored and fault labels masked from detector and codec developers; externally held-out authorized MRI where approval and source documentation permit.',
   'analysis_unit':'Person for uncertainty and cohort status; imaging-study and observation for reconstruction and event-level error.',
   'training_policy':'Train plan selection only on Phase35 training people; do not tune or reselect on Phase35 holdout or future externally authored challenge labels.',
   'synthetic_MCI_status':'Not attested. Source-proxy eligibility is not a clinical MCI diagnosis.',
   'source_procedure_gap':'Original Phase33 19-person FHIR has zero Procedure resources. Never borrow Phase29 manufactured procedure fixtures as their clinical source evidence.'},
 'arms':{
   'A':{'definition':'Faithful FHIR R4 → OMOP v5.5 plus pinned OHDSI ImageWG MI-CDM mapping with source-supported provenance and verified vocabulary.',
        'availability':'BLOCKED until exact frozen Athena CONCEPT.csv, independent review of 19 binding keys, verified source Procedure or explicitly manufactured source-tagged synthetic fixture policy, demographic missingness handling, and official target field and event-field validation.',
        'comparability':'Do not count as completed unless mandatory official schema fields, source-target cardinality, linkage, SQL replay and cohort equivalence all pass.'},
   'B':{'definition':'Prespecified value-only ablation from the same source observations, retaining only an opaque observation key, scalar numeric volume and declared allowed fields.',
        'availability':'Existing Phase34 synthetic shadow only. Generate a true target-matched baseline after valid Arm A; document its purposely missing laterality/timepoint/provenance.'},
   'C':{'definition':'Original frozen generic TFL v15 (unmodified baseline; distinguish separately versioned port if one becomes necessary).',
        'availability':'NONPORTABLE_TO_MRI_AS_IS; not executed in an imaging target experiment. Never substitute rewritten imaging rules under the frozen baseline label.'},
   'D':{'definition':'Phase35 adaptive minimal evidence carrier: select minimum metadata bytes among prespecified exact-roundtrip plans; VISIT_CODED training-selected; HYBRID_CODED fallback for ambiguous visits; quarantine contradictory source UID evidence.',
        'availability':'SYNTHETIC_TRANSPORT_PROTOTYPE ONLY; apply to same eligible source rows as A/B/C when those arms become executable.'}},
 'locked_objectives':[
   {'id':'H1','estimand':'Selected-field exact reconstruction of all 19 canonical source manifest fields under the D codec, independently checked against original raw FHIR.',
    'success_definition':'100% exact roundtrip for each non-quarantined eligible observation; report any mismatched field and denominator, with quarantined inputs separately.',
    'scope':'Engineering data integrity; not biological or clinical truth.'},
   {'id':'H2','estimand':'Canonical UTF-8 JSON metadata-byte change between D and inline representation on the same eligible observations.',
    'calculation':'100*(inline_bytes - selected_bytes)/inline_bytes, including codebooks and headers; compute per split and scenario, exclude quarantined inputs with explicit counts.',
    'scope':'Only directly comparable encodings; do not generalize current synthetic percentage to real MRI.'},
   {'id':'H3','estimand':'Future person-level source-versus-official target cohort agreement and false inclusions/exclusions after validated A and matched B/C/D.',
    'status':'GATED_NO_VALID_OFFICIAL_TARGET_YET','reference':'Prespecified source clinical cohort only when independently attested; synthetic proxy reported separately.'},
   {'id':'H4','estimand':'Prospective independently authored masked corruption detection by case and by event on unseen people.',
    'status':'GATED_NO_INDEPENDENT_AUTHOR_YET','definitions':'TP=correct flagged injected event, FP=alert not matching any injected event, FN=unflagged injected event; clean-case false alert rate reported separately.'}],
 'metrics_and_denominators':{
   'manifest':'observations with exact verified original raw-FHIR source; 18 observable fields checked per record plus derived opaque key, not 19 independent raw-source fields.',
   'reconstruction':'exact_restored_eligible_obs / all_nonquarantined_eligible_obs; report per-field mismatches, full-record failures, and all quarantines.',
   'metadata':'Count canonical JSON UTF-8 bytes for both plans with full shared dictionaries, keys, headers and metadata. Do not omit common overhead selectively.',
   'cohort':'TP, FP, FN, TN per person versus independently attested enrollment truth; source-only synthetic proxy distinct from diagnosis.',
   'reliability':'Report any TRACE/PASS or QA state preservation as a source metadata attribute, never infer new biological truth from the state.',
   'negative_controls':'Study UID collisions, multiple studies per visit, changed laterality, wrong unit, missing provenance, changed timepoint and fabricated value-only cohort inclusion.'},
 'analysis_plan':{
   'design':'Use paired source rows and matched participant sets across executable arms; preserve identities in secure private input, publish only synthetic or permitted aggregate outputs.',
   'uncertainty':'If independent sample is sufficiently large, 2,000 person-cluster paired bootstrap replicates for 95% percentile intervals. Report the number of independent people and events, never treat observations as independent subjects.',
   'multiplicity':'Before outcome access, declare one primary future outcome and arm contrasts in an amended sealed protocol; do not perform undisclosed post-hoc selection or claim confirmatory significance on the Phase35 synthetic benchmark.',
   'sample_size':'No sample-size or power calculation established yet. Complete before real-data recruitment and future confirmatory analyses.',
   'missing_data':'Pre-report absent baseline or follow-up hemisphere, invalid units, missing source gender/procedure and conflicting UIDs; do not silently impute demographic or imaging source facts.',
   'reporting':'Report failures, quarantined counts and all nonadmissible arms as NOT_EVALUABLE, not zero error or perfect performance.'},
 'leakage_and_claim_controls':[
   'Never give injected fault or cohort labels to the detector while scoring blinded cases.',
   'Do not call developer-generated holdouts independently externally validated.',
   'Do not use 4 manufactured Phase29 Procedures for the independent Phase33 19-person cohort.',
   'Do not equate exact manifest roundtrip with faithful raw-FHIR evidence unless original source crosswalk passes.',
   'Do not call this protocol public prospective preregistration or call novelty/clinical impact established.',
   'No official MI-CDM validation, actual pyOMOP ETL, Athena approval or real ADNI inference without direct evidence.'],
 'blocking_gates':{
   'frozen_private_athena_present':False,'original_reviewed_bindings':0,'original_binding_keys':19,
   'original_vocabulary_gap_instances':36,'phase33_source_procedure_count':0,
   'arm_A_validated':False,'arm_C_executed':False,'real_MRI_rows':0,'official_four_arm_executed':False,
   'official_target_cohort_equivalence':False,'external_independent_blinding':False},
 'standards_pins':{'imagewg_commit':IMAGEWG,'omop55_commit':OMOP55},
 'archiving':'Export and hash this experiment contract before obtaining any NEW independent experimental outcomes; retain historical Phase34/35 files unchanged.'
}
print('Registered arms: A blocked, B shadow, C nonportable, D synthetic prototype.')

Registered arms: A blocked, B shadow, C nonportable, D synthetic prototype.


## 6 · Fail-closed tests and public-safe exports
All generated public files contain synthetic identifiers, summary states and hashes only; they contain neither raw/private Athena concepts nor real patient data. No SQL target tables are loaded without source-supported bindings. Original FHIR and previous phase outputs remain unchanged.

In [16]:
TESTS36=[]
def test(name,cond,detail):
 TESTS36.append({'test':name,'passed':bool(cond),'detail':detail})
 assert cond,name+': '+detail

test('exact_phase35_artifact_hashes',all(sha(F35[k])==P35['artifact_sha256'][k] for k in REQ35),'7/7 original SHA-backed output files')
test('phase34_chain_hash',sha(J34)==P35['upstream']['phase34_shareable_sha256'],'Exact Phase35-to-34 SHA chain')
test('phase33_original_bundle_hash',sha(FHIR_PATH)==P34['upstream']['phase33_bundle_sha256'],'Exact Phase34-to-33 original FHIR')
test('raw_source_148_observation_keys',len(AUDIT)==148 and all(x['status']=='PASS' for x in AUDIT),'148 independent FHIR Observation matches')
test('selected_fields_2664_checks',FIELD_CHECK_COUNT==2664 and all(x['mismatch_count']==0 for x in AUDIT),'18 source-observable fields per synthetic row')
test('ten_extraction_negative_controls',len(NEGATIVE)==10 and all(x['passed'] for x in NEGATIVE),'Original FHIR unchanged')
test('source_observation_study_person_cardinality',len(E['Observation'])==148 and len(E['ImagingStudy'])==38 and len(E['Patient'])==19,'Phase33 synthetic cardinality')
test('source_procedure_absent_not_phase29_reused',len(E.get('Procedure',{}))==0 and len(PRO)==0,'Phase33 has no source procedures')
test('source_demographics_absent',all('gender' not in p and 'birthDate' not in p for p in E['Patient'].values()),'No invented gender or birthdate')
test('phase35_split_preserved',Counter(r['partition'] for r in SPLIT)=={'TRAIN':10,'HOLDOUT':9},'Person disjoint')
test('phase35_frozen_plan_and_14_tests',all(r['passed'].lower()=='true' for r in TESTS) and METHOD['version']=='P35_v1','No new plan search')
test('source_conflict_quarantined',next(x for x in STRESS if x['scenario']=='CONFLICTING_STUDY_UID_QUARANTINE')['deployed_plan']=='NONE','No compression of contradictory input')
test('phase35_six_case_benchmark_validated',len(brows)==6,'All prior stress cases enumerated')
test('public_binding_gates_not_silently_closed',len(BINDING)==19 and P35['gates']['original_vocab_gaps_blocked']==36,'19 unapproved binding keys, original 36 blocked')
test('target_schema_never_fabricated',not schema_present or len(FIELD_MATRIX)==81,'Missing original pinned CSV explicitly gates field claims')
test('official_arms_are_not_declared_executed',ARM[0]['status']=='BLOCKED' and ARM[2]['status']=='NONPORTABLE_NOT_EXECUTED','A/C/four-arm not executed')
test('internal_protocol_not_retroactively_preregistered',PROTOCOL['status']=='INTERNAL_VERSIONED_FUTURE_PROTOCOL_NOT_PUBLICLY_PREREGISTERED' and PROTOCOL['blocking_gates']['official_four_arm_executed'] is False,'Historical work is retrospective, future registered scope is clearly delimited')
assert len(TESTS36)==17 and all(x['passed'] for x in TESTS36)

OUTPUT={
 'ICHI2027_Phase36_SOURCE_FHIR_CROSSWALK.csv':(AUDIT,['source_observation_ref','field_checks','mismatch_count','status','mismatch_fields']),
 'ICHI2027_Phase36_NEGATIVE_EXTRACTION_TESTS.csv':(NEGATIVE,['test','mutated_field','expected','observed','passed']),
 'ICHI2027_Phase36_FROZEN_BENCHMARK.csv':(brows,['scenario','n_people','n_observations','status','deployed_plan','reduction_percent','oracle_comparison']),
 'ICHI2027_Phase36_TARGET_FIELD_EVIDENCE.csv':(FIELD_MATRIX,['table','field','required','source','evidence_status','synthetic_source_cohort','armA_field_populated']),
 'ICHI2027_Phase36_BINDING_ACTIONS.csv':(BINDING,['binding_key','phase31_public_status','phase35_binding_gate','required_action']),
 'ICHI2027_Phase36_ARM_READINESS.csv':(ARM,['arm','description','status','reason']),
 'ICHI2027_Phase36_INTEGRITY_TESTS.csv':(TESTS36,['test','passed','detail'])}
for filename,(rows,fields) in OUTPUT.items():csvsave(OUT_DIR/filename,rows,fields)
PROTOCOL_PATH=OUT_DIR/'ICHI2027_Phase36_EXPERIMENT_CONTRACT.json'
PROTOCOL_PATH.write_text(json.dumps(PROTOCOL,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
REPORT={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'36_independent_original_fhir_crosswalk_and_standards_arm_gate',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'privacy':'Public synthetic only, no private Athena concepts or real MRI/ADNI identifiers',
 'upstream':{'phase35_sha256':sha(J35),'phase34_sha256':sha(J34),
   'phase34_original_manifest_sha256':sha(SOURCE_PATH),'phase33_original_fhir_sha256':sha(FHIR_PATH),
   'imagewg_commit':IMAGEWG,'omop55_commit':OMOP55},
 'independent_source_audit':{'synthetic_people':19,'synthetic_imaging_studies':38,'synthetic_observations':148,
   'per_row_source_observable_fields':18,'exact_source_field_comparisons_passed':FIELD_CHECK_COUNT,
   'rows_exactly_reconciled':148,'prespecified_negative_controls_passed':sum(x['passed'] for x in NEGATIVE),
   'prespecified_negative_controls_total':10,'source_procedures_found':len(PRO),
   'source_gender_present':0,'source_birthdates_present':0,'real_DICOM_UID_attested':False},
 'phase35_frozen_replay':{'person_disjoint_train':10,'person_disjoint_holdout':9,
   'train_selected_plan':'VISIT_CODED','holdout_synthetic_metadata_reduction_percent':38.4897,
   'stress_scenarios_reconciled':6,'original_phase35_synthetic_tests_passing':14,
   'external_independence':False},
 'standards':{'official_pinned_imaging_inventory_read_back':bool(imaging_csv),
   'official_pinned_core_inventory_read_back':bool(core_csv),
   'original_phase29_36_vocabulary_gaps_still_blocked':True,
   'original_binding_keys':19,'original_binding_keys_approved_in_phase35':0,
   'phase33_new_cohort_source_procedure_gap_studies':38,
   'phase29_synthetic_procedures_transfer_to_phase33':False,
   'armA_loaded':False,'official_MICDM_loaded':False,'pyOMOP_executed':False,
   'four_arm_comparison':False,'real_MRI_rows':0,'MCI_cohort_equivalence':False},
 'checks':{'total':len(TESTS36),'passing':sum(x['passed'] for x in TESTS36)},
 'public_output_sha256':{name.replace('ICHI2027_Phase36_','').rsplit('.',1)[0].lower():sha(OUT_DIR/name) for name in OUTPUT} |
                       {'experiment_contract':sha(PROTOCOL_PATH)},
 'next_gate':'For true synthetic Arm A, add expressly synthetic source-linked Procedure test fixtures for this distinct 19-person Phase33 cohort and independently reviewed frozen Athena binding decisions; source gender/birthdate remain absent and must be handled transparently. Real MRI requires real attested source procedures/UIDs and authorized clinical data. No official target load yet.'
}
REPORT_PATH=OUT_DIR/'ICHI2027_Phase36_SHAREABLE.json'
REPORT_PATH.write_text(json.dumps(REPORT,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
assert all((OUT_DIR/name).is_file() and len(csvread(OUT_DIR/name))==len(rows) for name,(rows,_) in OUTPUT.items())
assert json.loads(REPORT_PATH.read_text())['checks']['passing']==17
assert sha(FHIR_PATH)==P34['upstream']['phase33_bundle_sha256']
print('Phase36 PASS:',len(TESTS36),'/',len(TESTS36),'controls; 148/148 raw FHIR crosswalk; 10/10 negative cases.')
print('Outputs: 7 CSVs, EXPERIMENT_CONTRACT JSON, SHAREABLE JSON saved:',OUT_DIR)
print('Upload SHAREABLE, SOURCE_FHIR_CROSSWALK, NEGATIVE_EXTRACTION_TESTS, TARGET_FIELD_EVIDENCE and ARM_READINESS.')

Phase36 PASS: 17 / 17 controls; 148/148 raw FHIR crosswalk; 10/10 negative cases.
Outputs: 7 CSVs, EXPERIMENT_CONTRACT JSON, SHAREABLE JSON saved: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs
Upload SHAREABLE, SOURCE_FHIR_CROSSWALK, NEGATIVE_EXTRACTION_TESTS, TARGET_FIELD_EVIDENCE and ARM_READINESS.


### Interpretation
- Exact raw-FHIR crosswalk and all negative controls passing would show that Phase 34/35 reconstructed metadata is faithful to the *selected original synthetic FHIR fields*. It would **not** establish anatomical, clinical, DICOM authenticity or standards-valid ETL.
- **Next prerequisite to actual Arm A:** private reviewed Athena concept mappings **and source-supported required fields**. In this 19-person cohort, all 38 imaging studies lack FHIR Procedure records; the unrelated 4 manufactured Phase 29 fixtures cannot supply them. The source also lacks gender and birthdates. These are independent blockers, not simply vocabulary issues.
- Independent externally authored controls and eligible real MRI remain necessary to evaluate generalization and research utility.